In [3]:
import spacy, spacy_udpipe
from spacy import displacy
import en_core_web_sm
import ru_core_news_sm
class Bot:
    def __init__(self, lang, raw):
        self.lang=lang
        self.raw=raw
    def preprosess(self):
        nlp = spacy.load(self.lang)
        doc=nlp(self.raw)
        return doc
    def tokenize(self):
        doc=self.preprosess()
        tokens = [token.text for token in doc]
        print(tokens)
        return tokens
    def sentenize(self):
        nlp = spacy.load(self.lang)
        doc=nlp(self.raw)
        nlp.add_pipe('sentencizer')
        sents = [sent.text for sent in doc.sents]
        return sents    
    def morph(self):
        tokens=''
        for token in self.preprosess():
            tokens+=f'{token.i:2}. {token.text:15} Lemma: {token.lemma_:15} POS: {token.pos_:6} Feats: {token.morph:5} SyntR: {token.dep_:9} Head: {token.head.text}\n'
        return tokens
    def dep_tree(self):
        doc = self.preprosess()
        return displacy.serve(doc, style='dep')
    def ents(self):
        doc=self.preprosess()
        entss=''
        for ent in doc.ents:
            entss+=f'Entity: {ent.text:30} Label: {ent.label_}\n'
        return entss
    def lemms(self):
        nlp = spacy.load(self.lang)
        doc=nlp(self.raw)
        lemmatizer = nlp.get_pipe("lemmatizer")
        lemms=''
        for token in doc:
            lemms+=f'{token.text}    {token.lemma_}\n'
        return lemms
     

In [4]:
# https://pytba.readthedocs.io/ru/latest/sync_version/index.html
import telebot
from telebot import types
bot=telebot.TeleBot('6386112080:AAGAQGGXACKJe8JWrdj6bob-YlFXv3PM7gQ')
@bot.message_handler(commands=['start'])
def langs(message):
    markup=types.InlineKeyboardMarkup()
    markup.add(types.InlineKeyboardButton('eng', callback_data='en'))
    markup.add(types.InlineKeyboardButton('ru', callback_data='ru'))
    bot.send_message(message.chat.id, 'choose lang', reply_markup=markup) 

def text_file(mes):
    if mes.content_type=='text':
        raw=mes.text
        return raw
    elif mes.content_type=='document':
        file_info = bot.get_file(mes.document.file_id)
        downloaded_file = bot.download_file(file_info.file_path)
        file = r"C:\Users\lizao\Desktop\smth\бот" + mes.document.file_name
        with open(f, 'r') as f:
            raw=file.read()
            return raw

def output(message, result):
    markup3=types.InlineKeyboardMarkup()
    markup3.add(types.InlineKeyboardButton('text', callback_data='text'))
    markup3.add(types.InlineKeyboardButton('file.txt', callback_data='txt'))
    markup3.add(types.InlineKeyboardButton('file.doc', callback_data='doc'))
    ans=bot.send_message(message.chat.id, 'choose format', reply_markup=markup3)

def output_morph(message):
    markup3=types.InlineKeyboardMarkup()
    markup3.add(types.InlineKeyboardButton('text', callback_data='text'))
    markup3.add(types.InlineKeyboardButton('file.txt', callback_data='txt'))
    markup3.add(types.InlineKeyboardButton('file.conllu', callback_data='conllu'))
    bot.send_message(message.chat.id, 'choose format', reply_markup=markup3)
    
def tok_sent(callback):
    markup2=types.InlineKeyboardMarkup()
    tok=types.InlineKeyboardButton('tokenize', callback_data='tokenize')
    sent=types.InlineKeyboardButton('sentenize', callback_data='sentenize')
    lemm=types.InlineKeyboardButton('lemmatization', callback_data='lemmatizatiion')
    markup2.row(tok, sent, lemm)
    morph=types.InlineKeyboardButton('morphoparsing', callback_data='morphoparsing')
    dep=types.InlineKeyboardButton('dependence tree', callback_data='dependence tree')
    ents=types.InlineKeyboardButton('entities', callback_data='entities')
    markup2.row(morph, dep, ents)
    bot.send_message(callback.message.chat.id, 'what to do', reply_markup=markup2)

def pr(mes):
    return mes.text

@bot.callback_query_handler(func=lambda callback: True)
def callback_mes(callback):
    if callback.data=='en':
        global lang
        lang="en_core_web_sm"
        tok_sent(callback)
    elif callback.data=='ru':
        lang="ru_core_news_sm"
        tok_sent(callback)
    elif callback.data=='tokenize': 
        ans=bot.send_message(callback.message.chat.id, 'send file of text')
        #raw=bot.register_for_reply(callback.message, )
        raw=bot.register_next_step_handler(ans, text_file)
        tokenize(lang, raw, callback.message) #запихнуть сюда message
    elif callback.data=='sentenize': 
        bot.send_message(callback.message.chat.id, 'send file of text')
        bot.register_next_step_handler_by_chat_id(callback.message.chat.id, text_file)
    elif callback.data == 'lemmatization':
        bot.send_message(callback.message.chat.id, 'send file of text')
        ...
    elif callback.data == 'morphoparsing':
        bot.send_message(callback.message.chat.id, 'send file of text')
        ...
    elif callback.data == 'dependence tree':
        bot.send_message(callback.message.chat.id, 'send file of text')
        ...
    elif callback.data == 'entities':
        bot.send_message(callback.message.chat.id, 'send file of text')
        ...
    elif callback.data == 'text':
        ...

def tokenize(lang, raw, message): # и сюда запихнуть message
    b=Bot(lang, raw)
    tokens=b.tokenize()
    #bot.send_message(mes.chat.id, )
    output(message) # чтобы выпихнуть его тут
def sentenize(lang, mes):
    b=Bot(lang, mes)
    sents=b.sentenize()
    bot.send_message(mes.chat.id, f'{sents}')
def lemmatize(lang, mes):
    b=Bot(lang, mes)
    lemms=b.lemms()
    bot.send_message(mes.chat.id, lemms)
def morphoparsing(lang, mes):
    b=Bot(lang, mes)
    m=b.morph()
    bot.send_message(mes.chat.id, m)
def dependence(lang, mes):
    b=Bot(lang, mes)
    #понять, что получаем
def entities(lang, mes):
    b=Bot(lang, mes)
    ents=b.ents()
    b.send_message(mes.chat.id, ents)
    
bot.infinity_polling()


2023-07-28 08:01:34,917 (__init__.py:960 MainThread) ERROR - TeleBot: "Infinity polling exception: [E1041] Expected a string, Doc, or bytes as input, but got: <class 'NoneType'>"
2023-07-28 08:01:34,919 (__init__.py:962 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "c:\Users\lizao\AppData\Local\Programs\Python\Python310\lib\site-packages\telebot\__init__.py", line 955, in infinity_polling
    self.polling(non_stop=True, timeout=timeout, long_polling_timeout=long_polling_timeout,
  File "c:\Users\lizao\AppData\Local\Programs\Python\Python310\lib\site-packages\telebot\__init__.py", line 1043, in polling
    self.__threaded_polling(non_stop=non_stop, interval=interval, timeout=timeout, long_polling_timeout=long_polling_timeout,
  File "c:\Users\lizao\AppData\Local\Programs\Python\Python310\lib\site-packages\telebot\__init__.py", line 1118, in __threaded_polling
    raise e
  File "c:\Users\lizao\AppData\Local\Programs\Python\Python310\lib\sit